## 数据清洗 -> 聚合 -> 画图

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# Matplotlib 全局中文显示设置
# =========================
plt.rcParams["font.sans-serif"] = [
    "PingFang SC",  # macOS 常见中文字体
    "Hiragino Sans GB",  # macOS 常见中文字体
    "Microsoft YaHei",  # Windows 常见中文字体
    "SimHei",  # Windows 常见中文字体
    "Arial Unicode MS"  # 兜底字体
]
plt.rcParams["axes.unicode_minus"] = False  # 解决负号显示异常

# =========================
# 1) 构造“脏数据”
# =========================
np.random.seed(42)  # 固定随机种子，保证复现
n = 120  # 数据条数（120天）

df = pd.DataFrame({
    "日期": pd.date_range("2025-01-01", periods=n, freq="D"),  # 日期列
    "类别": np.random.choice(["A类", "B类", "C类"], size=n, p=[0.4, 0.35, 0.25]),  # 分类列
    "销售额": np.random.normal(300, 80, size=n).round(2),  # 销售额
    "成本": np.random.normal(180, 50, size=n).round(2),  # 成本
})

# 注入缺失值和异常值（模拟真实脏数据）
df.loc[np.random.choice(df.index, 8, replace=False), "销售额"] = np.nan
df.loc[np.random.choice(df.index, 5, replace=False), "成本"] = np.nan
df.loc[np.random.choice(df.index, 3, replace=False), "销售额"] = -50  # 异常值：负销售额

print("原始数据（前5行）：")
print(df.head(), "\n")
print("原始数据缺失值统计：")
print(df.isna().sum(), "\n")

# =========================
# 2) 数据清洗
# =========================

# clean.groupby("日期", as_index=False)：按“日期”分组，把同一天的所有行放在一组。
clean = df.copy()  # 复制一份清洗，保留原始数据

# 销售额不能为负，负值截断为0
clean["销售额"] = clean["销售额"].clip(lower=0)

# 用中位数填充缺失值（抗异常值能力更强）
clean["销售额"] = clean["销售额"].fillna(clean["销售额"].median())
clean["成本"] = clean["成本"].fillna(clean["成本"].median())

# 新增利润列
clean["利润"] = clean["销售额"] - clean["成本"]

print("清洗后缺失值统计：")
print(clean.isna().sum(), "\n")

# =========================
# 3) 数据聚合
# =========================
# 按日期汇总（每日总销售额、总利润）
daily = clean.groupby("日期", as_index=False)[["销售额", "利润"]].sum()

# 按类别统计均值（平均销售额、平均利润）
by_cat = clean.groupby("类别", as_index=False)[["销售额", "利润"]].mean()

print("按类别均值：")
print(by_cat, "\n")

# =========================
# 4) 可视化
# =========================
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 左图：每日趋势折线图
axes[0].plot(daily["日期"], daily["销售额"], label="每日销售额")
axes[0].plot(daily["日期"], daily["利润"], label="每日利润")
axes[0].set_title("每日销售与利润趋势")
axes[0].set_xlabel("日期")
axes[0].set_ylabel("金额")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=45)

# 右图：各类别平均销售额柱状图
axes[1].bar(by_cat["类别"], by_cat["销售额"])
axes[1].set_title("各类别平均销售额")
axes[1].set_xlabel("类别")
axes[1].set_ylabel("平均销售额")

plt.tight_layout()
plt.show()